# Project 2: architecture comparison on small data
One notebook for analysis. Everything is local: it reads your existing `*_losses.json` and `*_model.pth` files, no re-export needed.

In [ ]:
import json, glob, os
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

# ---- CONFIG: local paths on your laptop (edit these) ----
LOG_DIR  = "."           # folder containing the *_losses.json files
CKPT_DIR = "."           # folder containing the *_model.pth files
SIZES = ["5M", "25M", "100M"]   # nested FineWeb-Edu subsets
TOKEN_BUDGET = 100e6            # every run trains on 100M tokens in total (6,079 updates)
# Parameter counts printed by each training notebook ("Parameters: ...")
PARAMS = {"arch1_stack4": 38_617_088, "arch2_chain4": 38_604_800, "arch3_parallel": 36_511_744}
# Only if you ever run this on Colab instead: upload a zip of logs and unzip it here.

## 1. Load your existing `*_losses.json` files
Files look like `arch1_stack4_5M_losses.json`, `arch2_chain4_MODEL_2_25M_losses.json`. The architecture label, dataset size and optional seed (`..._s1_...`) come from the filename; parameter count is read from the matching `*_model.pth`.

In [ ]:
import re, glob, json
FNAME = re.compile(r"^(?P<prefix>.+?)(?:_(?:s|seed)(?P<seed>\d+))?_(?P<size>\d+M)_losses\.json$")
TOKENS = {"5M": 5e6, "25M": 25e6, "100M": 100e6}
STEMS = {}    # (arch label, size, seed) -> file stem, used later to find the .pth files

def _col(rows, *needles):
    keys = list(rows[0].keys())
    k = next((k for k in keys if any(n in k.lower() for n in needles)), None)
    return [r[k] for r in rows] if k else None

def _pick(d, *needles):
    for k, v in d.items():
        if isinstance(v, list) and v and all(isinstance(x, (int, float)) for x in v) \
           and any(n in k.lower() for n in needles):
            return v
    return None

def parse_losses(obj):
    """Understands: dict of lists, or list of per-eval dicts."""
    if isinstance(obj, list) and obj and isinstance(obj[0], dict):
        return _col(obj, "train"), _col(obj, "val"), _col(obj, "token", "step", "iter")
    if isinstance(obj, dict):
        return _pick(obj, "train"), _pick(obj, "val"), _pick(obj, "token", "step", "iter")
    return None, None, None

def count_params(path):
    """Unique parameters in the state dict (tied weights counted once)."""
    try:
        import torch
        sd = torch.load(path, map_location="cpu", weights_only=True)
        if isinstance(sd, dict) and "model_state_dict" in sd: sd = sd["model_state_dict"]
        seen, n = set(), 0
        for v in sd.values():
            if v.data_ptr() in seen: continue
            seen.add(v.data_ptr()); n += v.numel()
        return n
    except Exception as e:
        print("  (could not count params:", type(e).__name__, ")"); return float("nan")

def has_diverged(val, thresh=1.0):
    """True if val loss ever jumps more than `thresh` above its running minimum."""
    best = float("inf")
    for v in val:
        if v > best + thresh: return True
        best = min(best, v)
    return False

def load_logs(log_dir=LOG_DIR):
    recs = []
    for p in sorted(glob.glob(f"{log_dir}/*_losses.json")):
        m = FNAME.match(os.path.basename(p))
        if not m: continue
        label = re.sub(r"_MODEL_\d+$", "", m["prefix"]); size = m["size"]
        seed = int(m["seed"] or 0)                    # no suffix = seed 0; "_s1" or "_seed1" = seed 1, etc.
        stem = os.path.basename(p)[: -len(f"_{size}_losses.json")]      # e.g. arch1_stack4_s1
        STEMS[(label, size, seed)] = stem
        obj = json.load(open(p))
        tr, va, x = parse_losses(obj)
        if va is None:
            raise ValueError(f"{os.path.basename(p)}: could not find a validation-loss list. "
                             f"Top-level structure: {list(obj.keys()) if isinstance(obj, dict) else type(obj)}")
        if tr is None: tr = [float("nan")] * len(va)
        if x is None or len(x) != len(va):          # no usable x-axis: spread evenly over the dataset size
            x = list(np.linspace(TOKENS[size]/len(va), TOKENS[size], len(va)))
        recs.append(dict(arch=label, size=size, seed=seed, tokens_seen=x, train_loss=tr, val_loss=va,
                         n_params=PARAMS.get(label) or count_params(f"{CKPT_DIR}/{stem}_{size}_model.pth"),
                         final_val=(obj.get('final_val', va[-1]) if isinstance(obj, dict) else va[-1]),
                         diverged=has_diverged(va),
                         train_seconds=float("nan")))
    return recs

## 2. Build the summary table

In [ ]:
recs = load_logs()
rows = []
for r in recs:
    ft = next((v for v in reversed(r["train_loss"]) if v == v), float("nan"))
    rows.append(dict(arch=r["arch"], size=r["size"], seed=r["seed"], final_train=ft,
                     final_val=r["final_val"], eval_val_last=r["val_loss"][-1],
                     best_eval_val=min(r["val_loss"]), n_params=r["n_params"], diverged=r["diverged"]))
df = pd.DataFrame(rows)
df["ppl"] = np.exp(df.final_val)
df["gap"] = df.eval_val_last - df.final_train      # same estimator on both sides (last periodic eval)
df["size"] = pd.Categorical(df["size"], SIZES, ordered=True)
ARCHS = sorted(df.arch.unique())
print(ARCHS, len(df), "runs;", int(df.diverged.sum()), "diverged:", df[df.diverged][["arch","size"]].values.tolist())
df.sort_values(["size","final_val"])

## 3. Headline: validation loss curves
Each run trains for the same 100M-token budget (6,079 updates), so the 5M subset is repeated about 20 times and the 25M subset about 4 times. Dashed = a run that diverged. Note the curves come from the periodic eval (first 50 batches of the val split); the headline numbers in the matrix use the full validation set (`final_val`).

In [ ]:
fig, axes = plt.subplots(1, len(SIZES), figsize=(5*len(SIZES), 4), sharey=True)
colors = dict(zip(ARCHS, plt.cm.tab10.colors))
for ax, size in zip(np.atleast_1d(axes), SIZES):
    for r in recs:
        if r["size"] != size: continue
        ax.plot(r["tokens_seen"], r["val_loss"], color=colors[r["arch"]], alpha=.9,
                ls="--" if r["diverged"] else "-", label=r["arch"])
    h, l = ax.get_legend_handles_labels()
    ax.legend(dict(zip(l, h)).values(), dict(zip(l, h)).keys(), fontsize=8)
    ep = round(TOKEN_BUDGET/TOKENS[size])
    ax.set_title(f"{size} unique tokens (~{ep} epoch{'s' if ep > 1 else ''})")
    ax.set_xlabel("training tokens seen"); ax.grid(alpha=.3)
    ax.set_ylim(min(min(r["val_loss"]) for r in recs) - .05, 8)   # first eval (random init) is ~11 and would squash the rest
np.atleast_1d(axes)[0].set_ylabel("validation loss (eval subset)")
plt.tight_layout(); plt.savefig("fig_val_curves.png", dpi=200); plt.show()

## 4. Results matrix: final val loss and perplexity (full validation set)
Diverged runs are shown grey with their value and excluded from the colour scale and the 'best' marker. arch2 shows its latest attempt; section 6b shows both attempts.

In [ ]:
agg = df.groupby(["arch","size"], observed=True).agg(
    val=("final_val","mean"), ppl=("ppl","mean"), gap=("gap","mean"),
    params=("n_params","first"), diverged=("diverged","max")).reset_index()
M = agg.pivot(index="arch", columns="size", values="val")[SIZES]
P = agg.pivot(index="arch", columns="size", values="ppl")[SIZES]
D = agg.pivot(index="arch", columns="size", values="diverged")[SIZES].fillna(False).astype(bool)
ok = M.where(~D)
lo, hi = np.nanmin(ok.values), np.nanmax(ok.values)
cmap = plt.cm.viridis_r.copy(); cmap.set_bad("#d0d0d0")
fig, ax = plt.subplots(figsize=(1.8*len(SIZES)+2, .8*len(M)+1.5))
im = ax.imshow(ok.values, cmap=cmap, vmin=lo, vmax=hi)
ax.set_xticks(range(len(SIZES)), SIZES); ax.set_yticks(range(len(M)), M.index)
for i in range(M.shape[0]):
    for j in range(M.shape[1]):
        v = M.values[i,j]
        if np.isnan(v): continue
        if D.values[i,j]:
            ax.text(j, i, f"diverged\n({v:.2f})", ha="center", va="center", color="k", fontsize=9); continue
        col = "k" if (v - lo) / (hi - lo + 1e-9) < .45 else "w"   # viridis_r: low loss = bright
        ax.text(j, i-.1, f"{v:.3f}", ha="center", va="center", color=col, fontsize=12, fontweight="bold")
        ax.text(j, i+.12, f"ppl {P.values[i,j]:.0f}", ha="center", va="center", color=col, fontsize=9)
        if v == np.nanmin(ok.values[:,j]):
            ax.text(j, i+.34, "best", ha="center", va="center", color=col, fontsize=8, fontweight="bold")
ax.set_title("Final val loss and perplexity, full val set (lower is better)", fontsize=10)
plt.colorbar(im); plt.tight_layout(); plt.savefig("fig_matrix.png", dpi=200); plt.show()
agg.round(4)

## 5. Overfitting gap (val − train at last eval)
How much worse each model does on unseen text than on training text. Diverged runs are excluded.

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4))
w = .8/len(ARCHS)
for k, a in enumerate(ARCHS):
    s = agg[(agg.arch==a) & (~agg.diverged)].set_index("size").reindex(SIZES)
    ax.bar(np.arange(len(SIZES)) + k*w, s["gap"], w, label=a, color=colors[a])
ax.set_xticks(np.arange(len(SIZES)) + w*(len(ARCHS)-1)/2, SIZES)
ax.set_ylabel("val loss − train loss"); ax.grid(alpha=.3); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig("fig_gap.png", dpi=200); plt.show()

## 6. Efficiency: validation loss vs parameter count

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4))
mk = {"5M": "o", "25M": "s", "100M": "^"}
for a in ARCHS:
    for _, r in agg[(agg.arch==a) & (~agg.diverged)].iterrows():
        ax.scatter(r.params/1e6, r.val, color=colors[a], marker=mk[r["size"]], s=70,
                   label=a if r["size"] == SIZES[0] else None)
for s_, m_ in mk.items(): ax.scatter([], [], color="gray", marker=m_, label=f"{s_} tokens")
ax.set_xlabel("parameters (M)"); ax.set_ylabel("final val loss (full val)"); ax.grid(alpha=.3)
ax.legend(fontsize=8); plt.tight_layout(); plt.savefig("fig_efficiency.png", dpi=200); plt.show()

## 6b. arch2_chain4: both training attempts
The retrain overwrote the files from your first arch2 run, so those numbers are embedded below, copied from that run's printed log (micro-batch 8 × accum 2, non-fused AdamW). Attempt 2 is read from the files in this folder (micro-batch 4 × accum 4, fused AdamW, as for arch1 and arch3). Same seed, data order and initialization both times.

In [ ]:
# ---- attempt 1: copied from the first run's printed log (final_val = full validation set) ----
FIRST_ATTEMPT = {
    "5M": dict(final_val=6.227, tokens_M=[0.0, 3.3, 6.6, 9.9, 13.2, 16.4, 19.7, 23.0, 26.3, 29.6, 32.9, 36.2, 39.5, 42.8, 46.1, 49.3, 52.6, 55.9, 59.2, 62.5, 65.8, 69.1, 72.4, 75.7, 79.0, 82.2, 85.5, 88.8, 92.1, 95.4, 98.7],
        val=[10.981, 7.248, 6.858, 6.679, 6.601, 6.523, 6.466, 6.436, 6.391, 6.359, 6.344, 6.34, 6.323, 6.308, 6.307, 6.304, 6.3, 6.296, 6.29, 6.29, 6.296, 6.289, 6.282, 6.281, 6.281, 6.282, 6.28, 6.278, 6.278, 6.278, 6.276],
        train=[10.981, 7.227, 6.792, 6.569, 6.431, 6.31, 6.19, 6.109, 6.003, 5.922, 5.859, 5.803, 5.746, 5.698, 5.65, 5.607, 5.582, 5.535, 5.519, 5.489, 5.465, 5.435, 5.409, 5.386, 5.37, 5.355, 5.344, 5.337, 5.323, 5.317, 5.307]),
    "25M": dict(final_val=9.617, tokens_M=[0.0, 3.3, 6.6, 9.9, 13.2, 16.4, 19.7, 23.0, 26.3, 29.6, 32.9, 36.2, 39.5, 42.8, 46.1, 49.3, 52.6, 55.9, 59.2, 62.5, 65.8, 69.1, 72.4, 75.7, 79.0, 82.2, 85.5, 88.8, 92.1, 95.4, 98.7],
        val=[10.981, 7.213, 6.812, 6.645, 6.535, 6.49, 6.428, 6.414, 6.375, 10.719, 9.58, 9.112, 9.732, 9.144, 8.854, 9.089, 9.09, 9.185, 9.36, 9.321, 9.655, 9.582, 9.496, 9.384, 9.356, 9.495, 9.462, 9.473, 9.506, 9.536, 9.569],
        train=[10.981, 7.248, 6.844, 6.663, 6.547, 6.496, 6.43, 6.409, 6.361, 10.67, 9.586, 9.126, 9.722, 9.136, 8.847, 9.088, 9.092, 9.176, 9.357, 9.309, 9.646, 9.569, 9.481, 9.367, 9.336, 9.479, 9.455, 9.451, 9.477, 9.521, 9.582]),
    "100M": dict(final_val=5.938, tokens_M=[0.0, 3.3, 6.6, 9.9, 13.2, 16.4, 19.7, 23.0, 26.3, 29.6, 32.9, 36.2, 39.5, 42.8, 46.1, 49.3, 52.6, 55.9, 59.2, 62.5, 65.8, 69.1, 72.4, 75.7, 79.0, 82.2, 85.5, 88.8, 92.1, 95.4, 98.7],
        val=[10.981, 7.207, 6.82, 6.65, 6.53, 6.448, 6.37, 6.312, 6.281, 6.259, 6.237, 6.196, 6.182, 6.156, 6.125, 6.109, 6.098, 6.081, 6.068, 6.056, 6.037, 6.03, 6.016, 6.016, 6.002, 5.993, 5.991, 5.987, 5.985, 5.984, 5.978],
        train=[10.981, 7.252, 6.857, 6.693, 6.564, 6.479, 6.407, 6.334, 6.294, 6.269, 6.242, 6.204, 6.184, 6.157, 6.121, 6.108, 6.094, 6.075, 6.055, 6.044, 6.017, 6.012, 5.999, 5.997, 5.986, 5.977, 5.971, 5.967, 5.968, 5.965, 5.959]),
}

def spike_at(tokens, val, thresh=1.0):
    """tokens (in millions) at the first eval where val loss jumps more than `thresh` above its running minimum."""
    best = float("inf")
    for t, v in zip(tokens, val):
        if v > best + thresh: return t
        best = min(best, v)
    return None

attempts = []
for size in SIZES:
    d = FIRST_ATTEMPT[size]
    attempts.append(dict(attempt=1, size=size, tokens_M=d["tokens_M"], val=d["val"], final_val=d["final_val"]))
    r = next((r for r in recs if r["arch"] == "arch2_chain4" and r["size"] == size), None)
    if r: attempts.append(dict(attempt=2, size=size, tokens_M=[t/1e6 for t in r["tokens_seen"]], val=r["val_loss"], final_val=r["final_val"]))

tbl = pd.DataFrame([dict(attempt=a["attempt"], size=a["size"], final_val=a["final_val"], ppl=round(float(np.exp(a["final_val"])), 1),
                         diverged=has_diverged(a["val"]), spike_at_Mtokens=spike_at(a["tokens_M"], a["val"])) for a in attempts])
print(f"{int(tbl.diverged.sum())} of {len(tbl)} arch2_chain4 runs diverged")
tbl

In [ ]:
fig, axes = plt.subplots(1, len(SIZES), figsize=(5*len(SIZES), 4), sharey=True)
for ax, size in zip(np.atleast_1d(axes), SIZES):
    for a in attempts:
        if a["size"] != size: continue
        ax.plot(a["tokens_M"], a["val"], color=colors["arch2_chain4"], ls="--" if a["attempt"] == 1 else "-",
                alpha=.6 if a["attempt"] == 1 else 1, label=f"attempt {a['attempt']} (final {a['final_val']:.2f})")
    ax.set_title(f"arch2_chain4, {size} unique tokens"); ax.set_xlabel("training tokens seen (millions)")
    ax.set_ylim(5.9, 10.5); ax.grid(alpha=.3); ax.legend(fontsize=8)
np.atleast_1d(axes)[0].set_ylabel("validation loss (eval subset)")
plt.tight_layout(); plt.savefig("fig_arch2_attempts.png", dpi=200); plt.show()

## 7. Same-prompt generations across architectures
The model definitions below are copied from your three training notebooks, with the same attribute names so the saved weights load directly. Generation uses temperature + top-k sampling with a fixed seed (your in-training samples use greedy decoding, which is why they loop).

In [ ]:
import torch, torch.nn as nn, tiktoken
device = "cuda" if torch.cuda.is_available() else "cpu"
tok = tiktoken.get_encoding("gpt2")
CFG = dict(vocab_size=50257, emb_dim=256, context_length=1028, num_heads=4,
           n_layers=8, n_attn=4, drop_rate=0.2, qkv_bias=False)
CONTEXT = CFG["context_length"]

class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, dropout, num_heads, qkv_bias=False):
        super().__init__()
        self.d_out, self.num_heads, self.head_dim = d_out, num_heads, d_out // num_heads
        self.w_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.w_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.w_values = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)
        self.outproj = nn.Linear(d_out, d_out)
    def forward(self, x):
        b, n, _ = x.shape
        k = self.w_key(x).view(b, n, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.w_values(x).view(b, n, self.num_heads, self.head_dim).transpose(1, 2)
        q = self.w_query(x).view(b, n, self.num_heads, self.head_dim).transpose(1, 2)
        ctx = torch.nn.functional.scaled_dot_product_attention(
            q, k, v, dropout_p=self.dropout.p if self.training else 0.0, is_causal=True).transpose(1, 2)
        return self.outproj(ctx.contiguous().view(b, n, self.d_out))

class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim)); self.shift = nn.Parameter(torch.zeros(emb_dim))
    def forward(self, x):
        m = x.mean(-1, keepdim=True); var = x.var(-1, keepdim=True, unbiased=False)
        return self.scale * (x - m) / torch.sqrt(var + self.eps) + self.shift

class Gelu(nn.Module):
    def forward(self, x): return torch.nn.functional.gelu(x)

class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layer = nn.Sequential(nn.Linear(cfg["emb_dim"], 4*cfg["emb_dim"]), Gelu(),
                                   nn.Linear(4*cfg["emb_dim"], cfg["emb_dim"]))
    def forward(self, x): return self.layer(x)

def _mha(cfg):
    return MultiHeadAttention(cfg["emb_dim"], cfg["emb_dim"], cfg["drop_rate"], cfg["num_heads"], cfg["qkv_bias"])

class StackBlock(nn.Module):        # arch1_stack4: 4 x (norm -> MHA -> residual), then FF
    def __init__(self, cfg):
        super().__init__()
        self.attns = nn.ModuleList([_mha(cfg) for _ in range(cfg["n_attn"])])
        self.attn_norms = nn.ModuleList([LayerNorm(cfg["emb_dim"]) for _ in range(cfg["n_attn"])])
        self.ff = FeedForward(cfg); self.ff_norm = LayerNorm(cfg["emb_dim"]); self.dropout = nn.Dropout(cfg["drop_rate"])
    def forward(self, x):
        for attn, norm in zip(self.attns, self.attn_norms):
            x = x + self.dropout(attn(norm(x)))
        return x + self.dropout(self.ff(self.ff_norm(x)))

class ChainBlock(nn.Module):        # arch2_chain4: one norm, 4 MHAs chained, one residual, then FF
    def __init__(self, cfg, n_attn=4):
        super().__init__()
        self.attns = nn.ModuleList([_mha(cfg) for _ in range(n_attn)])
        self.attn_norm = LayerNorm(cfg["emb_dim"])
        self.ff = FeedForward(cfg); self.ff_norm = LayerNorm(cfg["emb_dim"]); self.dropout = nn.Dropout(cfg["drop_rate"])
    def forward(self, x):
        h = self.attn_norm(x)
        for attn in self.attns: h = attn(h)
        x = x + self.dropout(h)
        return x + self.dropout(self.ff(self.ff_norm(x)))

class ParallelBlock(nn.Module):     # arch3_parallel: (MHA || FF) in parallel, then a second FF
    def __init__(self, cfg):
        super().__init__()
        self.norm1 = LayerNorm(cfg["emb_dim"]); self.attn = _mha(cfg); self.ff_par = FeedForward(cfg)
        self.norm2 = LayerNorm(cfg["emb_dim"]); self.ff_seq = FeedForward(cfg); self.dropout = nn.Dropout(cfg["drop_rate"])
    def forward(self, x):
        h = self.norm1(x)
        x = x + self.dropout(self.attn(h) + self.ff_par(h))
        return x + self.dropout(self.ff_seq(self.norm2(x)))

class Gptmodel(nn.Module):
    def __init__(self, cfg, block_cls):
        super().__init__()
        self.token_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_embed = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.dropout_layer = nn.Dropout(cfg["drop_rate"])
        self.transformer = nn.Sequential(*[block_cls(cfg) for _ in range(cfg["n_layers"])])
        self.final_LN = LayerNorm(cfg["emb_dim"])
        self.out_proj = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)
    def forward(self, ids):
        x = self.token_emb(ids) + self.pos_embed(torch.arange(ids.shape[1], device=ids.device))
        return self.out_proj(self.final_LN(self.transformer(self.dropout_layer(x))))

BLOCKS = {"arch1_stack4": StackBlock, "arch2_chain4": ChainBlock, "arch3_parallel": ParallelBlock}

_CACHE = {}
def load_model(arch, size, seed=0):
    key = (arch, size, seed)
    if key not in _CACHE:
        m = Gptmodel(CFG, BLOCKS[arch])
        m.load_state_dict(torch.load(f"{CKPT_DIR}/{STEMS[key]}_{size}_model.pth", map_location="cpu", weights_only=True))
        _CACHE[key] = m.to(device).eval()
        if len(_CACHE) > 3: _CACHE.pop(next(iter(_CACHE)))      # keep GPU memory small
    return _CACHE[key]

@torch.no_grad()
def generate(model, prompt, max_new=80, temperature=0.8, top_k=40, seed=123):
    g = torch.Generator(device=device).manual_seed(seed)
    ids = torch.tensor([tok.encode(prompt)], device=device)
    for _ in range(max_new):
        logits = model(ids[:, -CONTEXT:])[:, -1, :] / temperature
        v, _ = torch.topk(logits, top_k); logits[logits < v[:, [-1]]] = -float("inf")
        ids = torch.cat([ids, torch.multinomial(torch.softmax(logits, -1), 1, generator=g)], 1)
    return tok.decode(ids[0].tolist())

PROMPTS = ["The water cycle begins when", "In mathematics, a prime number is", "The French Revolution was caused by"]

# sanity check: parameter counts should match what your training notebooks printed
for a, blk in BLOCKS.items():
    n = sum(p.numel() for p in Gptmodel(CFG, blk).parameters())
    print(a, f"{n:,}", "OK" if n == PARAMS[a] else f"MISMATCH (expected {PARAMS[a]:,})")

In [ ]:
from IPython.display import HTML, display

GEN_SIZES = ["25M"]           # 25M is the only size where all three architectures trained without diverging; use SIZES to see every size
GEN_KW = dict(max_new=80, temperature=0.8, top_k=40, seed=123)    # same settings and seed for every model

def generation_table(size, prompts=PROMPTS, **kw):
    cols = {}
    for a in ARCHS:
        if (a, size, 0) not in STEMS: continue
        try:
            m = load_model(a, size)
        except FileNotFoundError as e:
            print("missing checkpoint, skipped:", e.filename); continue
        cols[a] = [generate(m, p, **kw) for p in prompts]
    return pd.DataFrame(cols, index=prompts)

def show_generations(sizes=GEN_SIZES, **kw):
    css = "<style>.gen td,.gen th{white-space:pre-wrap;vertical-align:top;text-align:left;max-width:420px;font-size:12px}</style>"
    for size in sizes:
        t = generation_table(size, **kw)
        t.to_csv(f"generations_{size}.csv")
        display(HTML(css + f"<h4>{size} unique tokens</h4>" + t.to_html(escape=True, classes="gen")))

show_generations(**GEN_KW)

## 8. Per-token loss on a held-out passage
Every token of one fixed passage is shaded by how surprised the model was by it (darker red = higher loss), one panel per architecture. The passage is cut from the validation region of your FineWeb file, starting at a document boundary. The first few tokens always look bad because the model has no context yet. One passage is an illustration, not statistical evidence: change `PASSAGE_OFFSET` to look at others.

In [ ]:
import html
import torch.nn.functional as F

TOKEN_FILE = r"D:\built a llm from scratch 124million parameters\fineweb-edu\fineweb_tokens.bin"   # path from your training notebooks
VAL_START = 3_000_000_000      # validation region used during training
PASSAGE_OFFSET, PASSAGE_LEN = 2_000_000, 200
VIEW_SIZE = "25M"              # which trained size to inspect (25M: the only size where all three architectures trained properly)

FALLBACK_TEXT = ("The water cycle describes how water moves between the oceans, the air and the land. "
                 "Heat from the sun makes water evaporate from oceans and lakes, and the vapour rises and cools into clouds. "
                 "When the droplets grow heavy enough they fall as rain or snow, and the water flows through rivers back to the sea, "
                 "where the cycle begins again.")

def val_passage_ids(offset=PASSAGE_OFFSET, n=PASSAGE_LEN):
    """n tokens of validation text starting right after an <|endoftext|> token, inside a document long enough to fill n."""
    arr = np.memmap(TOKEN_FILE, dtype=np.uint16, mode="r")
    w = np.asarray(arr[VAL_START + offset : VAL_START + offset + 50_000])
    bounds = np.flatnonzero(w == 50256)
    for lo, hi in zip(bounds[:-1] + 1, bounds[1:]):
        if hi - lo >= n: return [int(t) for t in w[lo:lo + n]]
    return [int(t) for t in w[:n]]          # no usable document boundary found: start anywhere

try:
    PASSAGE_IDS = val_passage_ids()
except Exception as e:
    print(f"Could not read the token file ({type(e).__name__}); using the fallback text. Set TOKEN_FILE above to use held-out text.")
    PASSAGE_IDS = tok.encode(FALLBACK_TEXT)
print(tok.decode(PASSAGE_IDS))

In [ ]:
@torch.no_grad()
def token_losses(model, ids):
    x = torch.tensor([ids], device=device)
    logits = model(x[:, :-1]).float()
    return F.cross_entropy(logits[0], x[0, 1:], reduction="none").cpu().numpy()

def _span(t, bg, tip):
    return f'<span title="{tip}" style="background:{bg}">{html.escape(tok.decode([t]))}</span>'

def _wrap(spans):
    return f'<div style="font-family:monospace;white-space:pre-wrap;line-height:1.9;font-size:13px">{"".join(spans)}</div>'

def loss_html(losses, ids, vmax=8):
    return _wrap([_span(t, f"rgba(220,50,50,{min(l / vmax, 1):.2f})", f"loss {l:.2f}") for t, l in zip(ids, losses)])

def diff_html(la, lb, ids, vmax=3):
    """red = model A more surprised than B, blue = B more surprised than A."""
    out = []
    for t, a_, b_ in zip(ids, la, lb):
        d = (a_ - b_) / vmax
        out.append(_span(t, f"rgba({'220,50,50' if d > 0 else '50,100,220'},{min(abs(d), 1):.2f})", f"A {a_:.2f} | B {b_:.2f}"))
    return _wrap(out)

targets = PASSAGE_IDS[1:]
LOSS = {a: token_losses(load_model(a, VIEW_SIZE), PASSAGE_IDS) for a in ARCHS if (a, VIEW_SIZE, 0) in STEMS}
display(HTML("".join(f"<h4>{a} &nbsp; mean loss on this passage: {l.mean():.3f}</h4>" + loss_html(l, targets)
                     for a, l in LOSS.items())))

### Where does one architecture beat another?
Pick two models. Red tokens: A was more surprised than B. Blue tokens: B was more surprised. The tables list the tokens with the biggest differences in each direction.

In [ ]:
A, B = "arch1_stack4", "arch3_parallel"      # change to compare any two
display(HTML(f"<h4>A = {A}, B = {B} (red: A worse, blue: B worse)</h4>" + diff_html(LOSS[A], LOSS[B], targets)))
tk = pd.DataFrame(dict(token=[tok.decode([t]) for t in targets], A=LOSS[A], B=LOSS[B]))
tk["A_minus_B"] = tk.A - tk.B
print(f"mean loss on this passage: A {LOSS[A].mean():.3f}, B {LOSS[B].mean():.3f}")
print("tokens where A is better (most negative A−B):"); display(tk.nsmallest(8, "A_minus_B").round(2))
print("tokens where B is better (most positive A−B):"); display(tk.nlargest(8, "A_minus_B").round(2))

## 9. Gradio demo (run last)

In [ ]:
# !pip -q install gradio
import gradio as gr
def demo_fn(arch, size, prompt, temperature, max_new):
    return generate(load_model(arch, size), prompt, int(max_new), temperature)

gr.Interface(demo_fn,
    [gr.Dropdown(ARCHS, label="Architecture"), gr.Dropdown(SIZES, label="Training tokens"),
     gr.Textbox(PROMPTS[0], label="Prompt"), gr.Slider(.2, 1.5, .8, label="Temperature"),
     gr.Slider(20, 200, 80, label="New tokens")],
    gr.Textbox(label="Output"), title="Architecture variants on small data").launch(share=True)